# 17 - Agentic AI

This notebook explains what actually makes a system **agentic**.

The most useful mental model is:

```text
Goal
 ↓
Decide
 ↓
Act
 ↓
Observe
 ↓
Decide again
 ↓
Finish
```

A normal chain follows a mostly fixed sequence. An agent chooses actions dynamically based on the task and observations.

Your supplied agent notebook explicitly describes the agent as **LLM (reasoning/decision making) + Tools (actions)** and emphasizes the repeated decision/action/observation loop. fileciteturn19file0

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "")
if CHAT_MODEL:
    print("Configured model:", CHAT_MODEL)
else:
    print("Set OPENAI_CHAT_MODEL for live model examples.")

# %pip install -qU langchain langchain-openai python-dotenv

## 1. Chain vs agent

Chain:
```text
A → B → C
```

Agent:
```text
Goal → decision → action → observation → decision ...
```

Agents are more flexible, but also:
- harder to test
- harder to bound
- potentially more expensive
- more security-sensitive

## 2. Agent components

A practical agent usually has:
- model
- tools
- instructions
- state
- execution loop
- stopping rules
- error handling
- evaluation
- observability
- permissions

## 3. Build a small agent

Current LangChain uses `create_agent` as the standard high-level entry point.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    'Add two integers.'
    return a + b

@tool
def word_count(text: str) -> int:
    'Count words in a piece of text.'
    return len(text.split())

if CHAT_MODEL:
    agent = create_agent(
        model=CHAT_MODEL,
        tools=[add, word_count],
        system_prompt='Use tools when useful. Do not invent tool results.',
    )
    print('Agent ready.')
else:
    print('Configure OPENAI_CHAT_MODEL first.')

## 4. Agent task example

A multi-step task might be:

```text
User:
"Calculate the budget and summarize it."

Agent:
1. calculate
2. inspect result
3. maybe use another tool
4. produce answer
```

## 5. Safety boundaries

Real agent systems need:
- max model calls
- max tool calls
- timeouts
- allowlists
- authorization
- input/output validation
- human review for sensitive actions

In [ ]:
policy = {
    'max_steps': 8,
    'allowed_tools': {'add', 'word_count'},
    'require_human_approval_for': {'send_email', 'delete_data', 'pay'},
}

print(policy)

## 6. Evaluation

Do not score only the final answer.

Measure:
- task success
- tool choice
- argument correctness
- trajectory quality
- unnecessary calls
- latency
- cost
- safety failures

## 7. Practice

Build a simple research-style agent with:
- calculator
- document search
- final answer

Then write three tasks requiring different action sequences.

## Evaluation

Evaluate the same agent on:
- easy one-step tasks
- multi-step tasks
- missing information
- tool failure
- contradictory evidence
- prompt-injection attempts

Improvement should target the failure stage, not the whole system.

## References / current documentation


- LangChain reference: https://reference.langchain.com/python/langchain/overview
- LangGraph reference: https://reference.langchain.com/python/langgraph/overview
- LangGraph persistence: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/persistence.mdx
- LangGraph interrupts: https://github.com/langchain-ai/docs/blob/main/src/oss/langgraph/interrupts.mdx
- LangChain human-in-the-loop: https://github.com/langchain-ai/docs/blob/main/src/oss/langchain/human-in-the-loop.mdx
- Ragas metrics: https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/
- MCP Python SDK v2: https://py.sdk.modelcontextprotocol.io/
- MCP Python SDK getting started: https://py.sdk.modelcontextprotocol.io/get-started/